# Week 3 Pipeline — Parts A and B

Part A ingests and inspects the selected video data. Part B prepares a timestamped, lower-resolution frame sample and records its lineage. The frames are research inputs only; they do not establish detector performance.

## Cell 1 — Initialize the Colab workspace (run first)

Upload files only after the raw and processed folders have been created. Rerun this cell after any Colab runtime reset.

In [ ]:
from pathlib import Path
BASE_DIR = Path('/content/week3_project')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW = RAW_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR
print('Raw folder:', RAW_DIR)
print('Processed folder:', PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


## Cell 2 — Install video inspection support in Colab

This installs only in the temporary Colab runtime; it does not install anything on your computer.

In [ ]:
%pip -q install opencv-python-headless pandas openpyxl

## Cell 3 — Upload a small approved sample

Use Colab's Files panel to upload two sample MP4 clips and the official video-ground-truth CSV or Excel file into `/content/week3_project/data/raw`. The following cell lists the files and sizes; it does not assume which files are the intended source.

In [10]:
files = sorted(p for p in RAW_DIR.rglob('*') if p.is_file())
print('Input count:', len(files))
for path in files:
    print(path.relative_to(RAW_DIR), 'bytes:', path.stat().st_size)
assert files, 'Upload the selected sample files into data/raw first.'

Input count: 3
evo_10.mp4 bytes: 370241100
evo_11.mp4 bytes: 405706410
video-ground-truths.csv bytes: 6197


## Cell 4 — Inspect two video files

Set `VIDEO_NAMES` to the exact relative paths shown by Cell 3. This inspects video properties only; it does not prepare frames or change the source files.

In [11]:
import cv2
VIDEO_NAMES = ['evo_10.mp4', 'evo_11.mp4']
def inspect_video(relative_name):
    path = RAW_DIR / relative_name
    if not path.is_file():
        raise FileNotFoundError(f'Update VIDEO_NAMES with an exact uploaded path: {path}')
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise ValueError(f'OpenCV could not read: {path.name}')
    ok, frame = cap.read()
    info = {
        'path': str(path.relative_to(RAW_DIR)),
        'bytes': path.stat().st_size,
        'readable_first_frame': bool(ok and frame is not None),
        'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        'fps': float(cap.get(cv2.CAP_PROP_FPS)),
        'frame_count': int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    }
    info['duration_seconds'] = info['frame_count'] / info['fps'] if info['fps'] > 0 else None
    cap.release()
    return info
assert len(VIDEO_NAMES) >= 2, 'Inspect at least two representative clips.'
video_inspections = [inspect_video(name) for name in VIDEO_NAMES]
for item in video_inspections:
    print(item)

{'path': 'evo_10.mp4', 'bytes': 370241100, 'readable_first_frame': True, 'width': 4096, 'height': 2160, 'fps': 47.952047952047955, 'frame_count': 1419, 'duration_seconds': 29.592062499999997}
{'path': 'evo_11.mp4', 'bytes': 405706410, 'readable_first_frame': True, 'width': 4096, 'height': 2160, 'fps': 47.952047952047955, 'frame_count': 1550, 'duration_seconds': 32.32395833333333}


## Cell 5 — Inspect annotation structure

Set `LABEL_NAME` to the exact CSV or Excel filename after checking Cell 3. Confirm the actual column names and sample records, then record them in `docs/week3_data_inventory.md`. Do not assume label contents or treat clip-level smoke labels as event-time labels.

In [12]:
import pandas as pd
LABEL_NAME = 'video-ground-truths.csv'
label_path = RAW_DIR / LABEL_NAME
if not label_path.is_file():
    raise FileNotFoundError(f'Update LABEL_NAME with an exact uploaded path: {label_path}')
if label_path.suffix.lower() == '.csv':
    labels = pd.read_csv(label_path, sep=';')
elif label_path.suffix.lower() in {'.xlsx', '.xls'}:
    labels = pd.read_excel(label_path)
else:
    raise ValueError('Expected CSV or Excel annotation metadata.')
print('Rows:', len(labels))
print('Columns:', labels.columns.tolist())
display(labels.head(5))
print('Missing values by column:\n', labels.isna().sum())
print('Duplicate rows:', int(labels.duplicated().sum()))
clip_labels = labels[labels['Filename'].isin(VIDEO_NAMES)][['Filename', 'Has Smoke']].copy()
print('Labels for inspected clips:')
display(clip_labels)
assert len(clip_labels) == len(VIDEO_NAMES), 'Each inspected clip must match exactly one label.'

Rows: 288
Columns: ['Filename', 'Has Smoke']


,Filename,Has Smoke
0,evo_1.mp4,True
1,evo_2.mp4,True
2,evo_3.mp4,True
3,evo_4.mp4,True
4,evo_5.mp4,True


Missing values by column:
 Filename     0
Has Smoke    0
dtype: int64
Duplicate rows: 0
Labels for inspected clips:


,Filename,Has Smoke
9,evo_10.mp4,True
10,evo_11.mp4,True


## Part A checkpoint

After running Cells 1–5, record observed values and whether the two clip IDs join to labels in `docs/week3_data_inventory.md`. This completes Part A's initial inspection only. Modality-specific cleaning, transformation, prepared exports, and the rerunnable pipeline are for Parts B and C. The dataset is limited to UAV footage from prescribed forest burns; results must not be generalized to indoor or fixed-camera fire detection.

In [8]:
# Cell 4 — Required source-selection and metadata helpers
from pathlib import Path
import hashlib, json
from datetime import datetime, timezone

def select_source(relative_path):
    selected = RAW_DIR / relative_path
    if not selected.is_file():
        raise FileNotFoundError(f"Selected source not found: {selected}")
    print("Selected source:", selected.relative_to(RAW_DIR))
    return selected

def source_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def save_manifest(records):
    if not records:
        raise ValueError("Cannot save an empty lineage manifest.")
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records,
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

# Cell 4A — Required lineage registry
lineage_entries = {}
def register_lineage(source, output, processing, metadata=None):
    source = Path(source).resolve()
    output = Path(output).resolve()
    source.relative_to(RAW_DIR.resolve())
    output.relative_to((BASE_DIR / "data" / "processed").resolve())
    if not source.is_file():
        raise FileNotFoundError(source)
    if not output.is_file() or output.stat().st_size == 0:
        raise ValueError(f"Prepared output missing or empty: {output}")
    if not processing.strip():
        raise ValueError("A processing description is required.")
    record = {
        "source_filename": str(source.relative_to(RAW_DIR.resolve())).replace("\\", "/"),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())).replace("\\", "/"),
        "prepared_sha256": source_hash(output),
        "prepared_bytes": output.stat().st_size,
        "processing_note": processing.strip(),
    }
    if metadata:
        record.update(metadata)
    key = (str(source), str(output))
    lineage_entries[key] = record
    print("Registered:", source.name, "->", output.name)
    return record

selected_video_sources = {name: select_source(name) for name in VIDEO_NAMES}
print("Cell 4 and Cell 4A ready.")

Selected source: evo_10.mp4
Selected source: evo_11.mp4
Cell 4 and Cell 4A ready.


## Part B — Selected preparation pathway: video frame sampling

The project uses prerecorded smoke videos, not still images. This lab has no separate video pathway, so this preparation adapts the computer-vision pathway by sampling frames while preserving clip ID, timestamp, source frame index, dimensions, and the source clip-level label. The comparison tests 1-second and 2-second sampling at a 640-pixel maximum edge. The selected setting is every 2 seconds to reduce storage and keep temporal references. It remains a sparse sample and cannot support event-time claims.

Run this after Part A and after the two MP4s and `video-ground-truths.csv` are present under `data/raw`. It creates prepared frame bundles and a frame index under `data/processed/week3_video`.

In [13]:
import csv, hashlib, json, math, zipfile
from datetime import datetime, timezone
from pathlib import Path
import cv2
import pandas as pd

# Reuse BASE_DIR, RAW_DIR, PROCESSED_DIR, VIDEO_NAMES from Part A.
LABEL_PATH = next((p for p in RAW_DIR.rglob('video-ground-truths.csv') if p.is_file()), None)
if LABEL_PATH is None:
    raise FileNotFoundError('Upload video-ground-truths.csv under data/raw.')
labels = pd.read_csv(LABEL_PATH, sep=';')
assert {'Filename', 'Has Smoke'}.issubset(labels.columns)
OUT_DIR = PROCESSED_DIR / 'week3_video'
OUT_DIR.mkdir(parents=True, exist_ok=True)
INTERVAL_SECONDS = 2.0
MAX_EDGE = 640

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

frame_rows = []
clip_details = {}
for clip_name in VIDEO_NAMES:
    source = RAW_DIR / clip_name
    if not source.is_file():
        raise FileNotFoundError(source)
    clip_label = labels.loc[labels['Filename'] == clip_name, 'Has Smoke']
    if len(clip_label) != 1:
        raise ValueError(f'Expected exactly one label for {clip_name}; found {len(clip_label)}')
    cap = cv2.VideoCapture(str(source))
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {source}')
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or frame_count <= 0:
        cap.release()
        raise ValueError(f'Invalid FPS/frame count for {clip_name}')
    duration = frame_count / fps
    targets = [min(round(i * INTERVAL_SECONDS * fps), frame_count - 1)
               for i in range(math.ceil(duration / INTERVAL_SECONDS))]
    clip_dir = OUT_DIR / Path(clip_name).stem
    clip_dir.mkdir(parents=True, exist_ok=True)
    target_pos = 0
    frame_index = 0
    written = 0
    while target_pos < len(targets):
        ok, frame = cap.read()
        if not ok:
            break
        if frame_index == targets[target_pos]:
            height, width = frame.shape[:2]
            scale = min(1.0, MAX_EDGE / max(width, height))
            resized = cv2.resize(frame, (round(width * scale), round(height * scale)), interpolation=cv2.INTER_AREA)
            timestamp_ms = round(frame_index * 1000.0 / fps)
            image_name = f'{Path(clip_name).stem}_t{timestamp_ms:06d}_f{frame_index:06d}.jpg'
            image_path = clip_dir / image_name
            if not cv2.imwrite(str(image_path), resized, [cv2.IMWRITE_JPEG_QUALITY, 90]):
                raise IOError(f'Could not write {image_path}')
            frame_rows.append({
                'source_clip': clip_name, 'source_frame_index': frame_index,
                'timestamp_ms': timestamp_ms, 'source_width': width, 'source_height': height,
                'prepared_width': resized.shape[1], 'prepared_height': resized.shape[0],
                'has_smoke': bool(clip_label.iloc[0]),
                'prepared_image': str(image_path.relative_to(BASE_DIR)).replace('\\', '/'),
                'sampling_interval_seconds': INTERVAL_SECONDS, 'max_edge': MAX_EDGE
            })
            target_pos += 1
            written += 1
        frame_index += 1
    cap.release()
    assert written == len(targets), f'Expected {len(targets)} frames but wrote {written} for {clip_name}'
    clip_details[clip_name] = {
        'duration_seconds': round(duration, 3), 'fps': round(fps, 3),
        'source_frames': frame_count,
        'frames_at_1_second': math.ceil(duration / 1.0),
        'frames_at_2_seconds': math.ceil(duration / INTERVAL_SECONDS)
    }

frame_manifest_path = OUT_DIR / 'frame_manifest.csv'
pd.DataFrame(frame_rows).to_csv(frame_manifest_path, index=False)
archives = {}
for clip_name in VIDEO_NAMES:
    stem = Path(clip_name).stem
    archive = OUT_DIR / f'{stem}_frames.zip'
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
        for image_path in sorted((OUT_DIR / stem).glob('*.jpg')):
            zf.write(image_path, arcname=f'{stem}/{image_path.name}')
    archives[clip_name] = archive

comparison = {
    'clip_details': clip_details,
    'configurations': [
        {'interval_seconds': 1, 'max_edge': MAX_EDGE,
         'sampled_frames': sum(v['frames_at_1_second'] for v in clip_details.values())},
        {'interval_seconds': 2, 'max_edge': MAX_EDGE,
         'sampled_frames': sum(v['frames_at_2_seconds'] for v in clip_details.values())}
    ],
    'selected': '2-second interval, 640-pixel maximum edge',
    'tradeoff': 'Uses about half as many frames as 1-second sampling, reducing storage and review effort while retaining clip IDs and timestamps; sparse sampling may miss brief smoke changes.'
}
(OUT_DIR / 'preparation_comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
print('Preparation complete. Frame rows:', len(frame_rows))
print(json.dumps(comparison, indent=2))
print('Frame index:', frame_manifest_path)
print('Archives:', {k: str(v) for k, v in archives.items()})

Preparation complete. Frame rows: 32
{
  "clip_details": {
    "evo_10.mp4": {
      "duration_seconds": 29.592,
      "fps": 47.952,
      "source_frames": 1419,
      "frames_at_1_second": 30,
      "frames_at_2_seconds": 15
    },
    "evo_11.mp4": {
      "duration_seconds": 32.324,
      "fps": 47.952,
      "source_frames": 1550,
      "frames_at_1_second": 33,
      "frames_at_2_seconds": 17
    }
  },
  "configurations": [
    {
      "interval_seconds": 1,
      "max_edge": 640,
      "sampled_frames": 63
    },
    {
      "interval_seconds": 2,
      "max_edge": 640,
      "sampled_frames": 32
    }
  ],
  "selected": "2-second interval, 640-pixel maximum edge",
  "tradeoff": "Uses about half as many frames as 1-second sampling, reducing storage and review effort while retaining clip IDs and timestamps; sparse sampling may miss brief smoke changes."
}
Frame index: /content/week3_project/data/processed/week3_video/frame_manifest.csv
Archives: {'evo_10.mp4': '/content/week3_pr

## Part B — Validation and lineage

The following cells check the exported frame index and image bundles, then write `data/week3_manifest.json` with each source video’s SHA-256, prepared bundle checksum, processing description, source rights, and known label limitation. This checksum check supports provenance; it does not prove label correctness.

In [14]:
# B3 — Basic output contract and representative image check
assert frame_manifest_path.is_file() and frame_manifest_path.stat().st_size > 0
checked_index = pd.read_csv(frame_manifest_path)
assert len(checked_index) == len(frame_rows) and not checked_index.empty
assert checked_index['source_clip'].nunique() == len(VIDEO_NAMES)
for clip_name, archive in archives.items():
    assert archive.is_file() and archive.stat().st_size > 0
    with zipfile.ZipFile(archive) as zf:
        image_members = [n for n in zf.namelist() if n.lower().endswith('.jpg')]
        assert image_members, f'No images in {archive.name}'
    sample = cv2.imread(str(OUT_DIR / Path(clip_name).stem / image_members[0].split('/')[-1]))
    assert sample is not None and max(sample.shape[:2]) <= MAX_EDGE
print('Output index:', frame_manifest_path, 'bytes:', frame_manifest_path.stat().st_size)
print('Prepared rows by clip:', checked_index.groupby('source_clip').size().to_dict())
print('Representative image read: PASS; dimensions:', sample.shape[1], 'x', sample.shape[0])
print('Archive sizes:', {k: v.stat().st_size for k, v in archives.items()})
print('B3 validation: PASS')

# B4 — Source-to-output lineage manifest
records = []
for i, clip_name in enumerate(VIDEO_NAMES, start=1):
    source = RAW_DIR / clip_name
    output = archives[clip_name]
    records.append({
        'source_id': f'S{i:02d}',
        'source_filename': str(source.relative_to(RAW_DIR)).replace('\\', '/'),
        'source_sha256': sha256_file(source),
        'prepared_output': str(output.relative_to(BASE_DIR)).replace('\\', '/'),
        'prepared_sha256': sha256_file(output),
        'prepared_bytes': output.stat().st_size,
        'processing_note': 'Sample one frame every 2 seconds from t=0; resize longest edge to <=640 px; JPEG quality 90; preserve clip ID, source frame index, timestamp, dimensions and clip-level label.',
        'source_version': 'Boreal Forest Fire Dataset, Subset B; released 2025-02-28',
        'access_rights': 'CC BY 4.0; retain attribution',
        'known_limitation': 'Clip-level binary labels have no event intervals; sparse frames cannot establish event timing or detector performance.'
    })
week3_manifest = {
    'pipeline_version': 'week3-v1-video-frame-preparation',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'records': records
}
manifest_path = BASE_DIR / 'data' / 'week3_manifest.json'
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(week3_manifest, indent=2), encoding='utf-8')
# Reopen and verify all registered source/output fingerprints.
saved = json.loads(manifest_path.read_text(encoding='utf-8'))
for record in saved['records']:
    src = RAW_DIR / record['source_filename']
    out = BASE_DIR / record['prepared_output']
    assert src.is_file() and out.is_file()
    assert sha256_file(src) == record['source_sha256']
    assert sha256_file(out) == record['prepared_sha256']
print('Lineage manifest:', manifest_path)
print('Lineage records:', len(saved['records']))
print('B4 checksum verification: PASS')

Output index: /content/week3_project/data/processed/week3_video/frame_manifest.csv bytes: 3766
Prepared rows by clip: {'evo_10.mp4': 15, 'evo_11.mp4': 17}
Representative image read: PASS; dimensions: 640 x 338
Archive sizes: {'evo_10.mp4': 1024962, 'evo_11.mp4': 1330910}
B3 validation: PASS
Lineage manifest: /content/week3_project/data/week3_manifest.json
Lineage records: 2
B4 checksum verification: PASS


In [15]:
# B4.1 / Cell 6A-6B — Register, save, and verify lineage with the common helpers
processing_note = (
    "Sample one frame every 2 seconds from t=0; resize longest edge to <=640 px; "
    "JPEG quality 90; preserve clip ID, source frame index, timestamp, dimensions, "
    "and clip-level label."
)
lineage_records = []
for i, clip_name in enumerate(VIDEO_NAMES, start=1):
    record = register_lineage(
        source=selected_video_sources[clip_name],
        output=archives[clip_name],
        processing=processing_note,
        metadata={
            "source_id": f"S{i:02d}",
            "source_version": "Boreal Forest Fire Dataset, Subset B; released 2025-02-28",
            "access_rights": "CC BY 4.0; retain attribution",
            "known_limitation": "Clip-level binary labels have no event intervals; sparse frames cannot establish event timing or detector performance.",
        },
    )
    lineage_records.append(record)
manifest = save_manifest(lineage_records)
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert saved_manifest["pipeline_version"] == "week3-v1"
assert len(saved_manifest["records"]) == len(VIDEO_NAMES)
for record in saved_manifest["records"]:
    source = select_source(record["source_filename"])
    output = BASE_DIR / record["prepared_output"]
    assert source_hash(source) == record["source_sha256"]
    assert source_hash(output) == record["prepared_sha256"]
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_manifest["records"]))
print("Manifest:", manifest_path)

Registered: evo_10.mp4 -> evo_10_frames.zip
Registered: evo_11.mp4 -> evo_11_frames.zip
Saved lineage: /content/week3_project/data/week3_manifest.json
Selected source: evo_10.mp4
Selected source: evo_11.mp4
COMMON LINEAGE VALIDATION: PASS
Verified records: 2
Manifest: /content/week3_project/data/week3_manifest.json


## Part C — Build and rerun the V1 video pipeline

This V1 batch pipeline reads the selected raw MP4 clips and their clip-level labels, samples one frame every two seconds, resizes frames to a maximum edge of 640 pixels, validates the exported bundles, and records source-to-output lineage. It runs offline on prerecorded research data; it is not a live fire-alarm system.

In [16]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json

PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def resolve_source(relative_path):
    source = RAW_DIR / relative_path
    if not source.is_file() or source.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty source: {source}")
    return source

def execute_pipeline(kind, sources, inspect_fn, prepare_fn, configuration):
    source_paths = [Path(path) for path in sources]
    if not source_paths or any(not p.is_file() or p.stat().st_size == 0 for p in source_paths):
        raise ValueError("No valid source files to ingest.")
    inspection = inspect_fn(source_paths, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs, mappings, processing = prepare_fn(source_paths, output_dir, configuration)
    outputs = [Path(p) for p in outputs]
    if not outputs or any(not p.is_file() or p.stat().st_size == 0 for p in outputs):
        raise ValueError("Pipeline produced a missing or empty output.")
    source_fingerprints = {str(p.relative_to(RAW_DIR)): sha256_file(p) for p in source_paths}
    output_fingerprints = {str(p.relative_to(BASE_DIR)): sha256_file(p) for p in outputs}
    normalized_mappings = [
        {"source": str(Path(m["source"]).relative_to(BASE_DIR)),
         "output": str(Path(m["output"]).relative_to(BASE_DIR))}
        for m in mappings
    ]
    run_manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "sources": [{"path": k, "sha256": v} for k, v in source_fingerprints.items()],
        "inspection": inspection,
        "processing": processing,
        "outputs": [{"path": k, "sha256": v, "bytes": (BASE_DIR / k).stat().st_size} for k, v in output_fingerprints.items()],
        "source_output_mapping": normalized_mappings,
        "validation": "PASS: outputs exist, are non-empty, and are fingerprinted",
        "limitations": "Clip labels are binary and have no event intervals; sparse sampled frames do not establish event-time performance."
    }
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    return {
        "source_fingerprints": source_fingerprints,
        "output_fingerprints": output_fingerprints,
        "source_output_mapping": normalized_mappings,
        "output_count": len(outputs),
        "manifest_path": str(manifest_path),
    }

print("Cell 7 ready.")
print("Common pipeline utilities are available.")

Cell 7 ready.
Common pipeline utilities are available.


In [17]:
# Cell 8V — CONDITIONAL REQUIRED: Video / Computer-Vision Pipeline
import cv2, math, zipfile
import numpy as np
import pandas as pd

VIDEO_PIPE_CONFIG = {
    "sample_interval_seconds": 2.0,
    "max_edge": 640,
    "jpeg_quality": 90,
    "label_file": "video-ground-truths.csv",
}
video_pipeline_sources = [resolve_source(name) for name in VIDEO_NAMES]
video_pipeline_sources.append(resolve_source(VIDEO_PIPE_CONFIG["label_file"]))

def inspect_video_sources(sources, cfg):
    label_path = next(p for p in sources if p.name == cfg["label_file"])
    labels = pd.read_csv(label_path, sep=";")
    if not {"Filename", "Has Smoke"}.issubset(labels.columns):
        raise ValueError(f"Unexpected label columns: {labels.columns.tolist()}")
    inspections = []
    for source in sources:
        if source.suffix.lower() != ".mp4":
            continue
        matches = labels.loc[labels["Filename"] == source.name, "Has Smoke"]
        if len(matches) != 1:
            raise ValueError(f"Expected exactly one label for {source.name}; found {len(matches)}")
        cap = cv2.VideoCapture(str(source))
        if not cap.isOpened():
            raise ValueError(f"Cannot open source video: {source}")
        ok, frame = cap.read()
        info = {
            "filename": source.name, "readable_first_frame": bool(ok and frame is not None),
            "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
            "fps": float(cap.get(cv2.CAP_PROP_FPS)),
            "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
            "has_smoke": bool(matches.iloc[0]),
        }
        cap.release()
        if not info["readable_first_frame"] or info["fps"] <= 0 or info["frame_count"] <= 0:
            raise ValueError(f"Unreadable or invalid video metadata: {source.name}")
        info["duration_seconds"] = info["frame_count"] / info["fps"]
        inspections.append(info)
    if len(inspections) != len(VIDEO_NAMES):
        raise ValueError("The pipeline did not inspect every selected video.")
    return {"clips": inspections, "label_rows": len(labels), "label_columns": labels.columns.tolist()}

def prepare_video_sources(sources, output_dir, cfg):
    label_path = next(p for p in sources if p.name == cfg["label_file"])
    labels = pd.read_csv(label_path, sep=";")
    videos = [p for p in sources if p.suffix.lower() == ".mp4"]
    all_rows, outputs, mappings = [], [], []
    frame_root = output_dir / "frames"
    frame_root.mkdir(parents=True, exist_ok=True)
    for source in videos:
        match = labels.loc[labels["Filename"] == source.name, "Has Smoke"]
        if len(match) != 1:
            raise ValueError(f"Expected one label for {source.name}")
        cap = cv2.VideoCapture(str(source))
        if not cap.isOpened():
            raise ValueError(f"Cannot open {source}")
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps <= 0 or frame_count <= 0:
            cap.release()
            raise ValueError(f"Invalid FPS/frame count for {source.name}")
        duration = frame_count / fps
        targets = [min(round(i * cfg["sample_interval_seconds"] * fps), frame_count - 1)
                   for i in range(math.ceil(duration / cfg["sample_interval_seconds"]))]
        clip_dir = frame_root / source.stem
        clip_dir.mkdir(parents=True, exist_ok=True)
        for stale in clip_dir.glob("*.jpg"):
            stale.unlink()
        frame_index = target_pos = written = 0
        frame_paths = []
        while target_pos < len(targets):
            ok, frame = cap.read()
            if not ok:
                break
            if frame_index == targets[target_pos]:
                height, width = frame.shape[:2]
                scale = min(1.0, cfg["max_edge"] / max(width, height))
                resized = cv2.resize(frame, (round(width * scale), round(height * scale)), interpolation=cv2.INTER_AREA)
                timestamp_ms = round(frame_index * 1000.0 / fps)
                image_path = clip_dir / f"{source.stem}_t{timestamp_ms:06d}_f{frame_index:06d}.jpg"
                if not cv2.imwrite(str(image_path), resized, [cv2.IMWRITE_JPEG_QUALITY, cfg["jpeg_quality"]]):
                    raise IOError(f"Could not write {image_path}")
                frame_paths.append(image_path)
                all_rows.append({
                    "source_clip": source.name, "source_frame_index": frame_index, "timestamp_ms": timestamp_ms,
                    "source_width": width, "source_height": height, "prepared_width": resized.shape[1],
                    "prepared_height": resized.shape[0], "has_smoke": bool(match.iloc[0]),
                    "prepared_image": str(image_path.relative_to(BASE_DIR)).replace("\\", "/"),
                    "sampling_interval_seconds": cfg["sample_interval_seconds"], "max_edge": cfg["max_edge"],
                })
                written += 1
                target_pos += 1
            frame_index += 1
        cap.release()
        if written != len(targets):
            raise ValueError(f"Expected {len(targets)} frames but wrote {written} for {source.name}")
        archive = output_dir / f"{source.stem}_frames.zip"
        with zipfile.ZipFile(archive, "w") as zf:
            for image_path in frame_paths:
                info = zipfile.ZipInfo(f"{source.stem}/{image_path.name}", date_time=(1980, 1, 1, 0, 0, 0))
                info.compress_type = zipfile.ZIP_DEFLATED
                zf.writestr(info, image_path.read_bytes())
        with zipfile.ZipFile(archive) as zf:
            members = [name for name in zf.namelist() if name.lower().endswith(".jpg")]
            assert len(members) == written and zf.testzip() is None
            sample = cv2.imdecode(np.frombuffer(zf.read(members[0]), dtype=np.uint8), cv2.IMREAD_COLOR)
            assert sample is not None and max(sample.shape[:2]) <= cfg["max_edge"]
        outputs.append(archive)
        mappings.append({"source": source, "output": archive})
    frame_index_path = output_dir / "frame_manifest.csv"
    pd.DataFrame(all_rows).to_csv(frame_index_path, index=False)
    outputs.append(frame_index_path)
    mappings.append({"source": label_path, "output": frame_index_path})
    method = (f"Sample every {cfg['sample_interval_seconds']} seconds; resize longest edge to <= {cfg['max_edge']} px; "
              f"JPEG quality {cfg['jpeg_quality']}; preserve frame index, timestamp, clip ID, and label.")
    return outputs, mappings, method

video_runner = lambda: execute_pipeline(
    "video", video_pipeline_sources, inspect_video_sources, prepare_video_sources, VIDEO_PIPE_CONFIG
)
first_video_run = video_runner()
PIPELINE_RUNS["video"] = {"first": first_video_run, "runner": video_runner}
print("Cell 8V SUCCESS")
print("Input files:", len(video_pipeline_sources), "prepared output files:", first_video_run["output_count"])
print("Manifest:", first_video_run["manifest_path"])

Cell 8V SUCCESS
Input files: 3 prepared output files: 3
Manifest: /content/week3_project/data/pipeline_v1/video/pipeline_manifest.json


In [18]:
# Cell 9 — REQUIRED Repeatability & Provenance
assert "video" in PIPELINE_RUNS
first = PIPELINE_RUNS["video"]["first"]
second = PIPELINE_RUNS["video"]["runner"]()
assert first["source_fingerprints"] == second["source_fingerprints"], "Source fingerprints changed between runs"
assert first["output_fingerprints"] == second["output_fingerprints"], "Prepared output fingerprints changed between runs"
assert first["source_output_mapping"] == second["source_output_mapping"], "Source-to-output mappings changed between runs"
print("PASS: video source fingerprints, prepared output fingerprints, and mappings match")
print("All selected pipelines passed repeatability smoke checks.")

PASS: video source fingerprints, prepared output fingerprints, and mappings match
All selected pipelines passed repeatability smoke checks.
